In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/mnist-digit-recognizer'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **loading my dataset**

In [ ]:
data=pd.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [ ]:
data


# **checking what shape of data we are working on**

In [ ]:
data.head()

In [ ]:
data = np.array(data)

In [ ]:
data.shape

# **to ensure consistency , shuffling the data**

In [ ]:
np.random.shuffle(data)

In [ ]:
data_dev=data[0:1000]

# **Transposing to transform the rows of data in numpy array as features in form of columns** # 

In [ ]:
Y_dev = data_dev[:, 0]  
X_dev = data_dev[:, 1:].T / 255.0

In [ ]:
X_dev
Y_dev

In [ ]:
data_train= data[1000:data.shape[0]].T
data_train

In [ ]:
Y_train = data_train[0]
X_train = data_train[1:data.shape[1]] / 255.0 

In [ ]:
X_train[:,0].shape

In [ ]:
def init_params():
    W1 = np.random.randn(128, 784) * np.sqrt(2/784)  # Xavier initialization
    B1 = np.zeros((128, 1))
    W2 = np.random.randn(10, 128) * np.sqrt(2/128)   # 10 output classes
    B2 = np.zeros((10, 1))
    return W1, B1, W2, B2  

In [ ]:
def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    # subtract max(Z) for numerical stability
    exp_vals = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_vals / np.sum(exp_vals, axis=0, keepdims=True)

def propogation_context_frwd(W1, B1, W2, B2, X):
    Z1 = W1.dot(X) + B1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + B2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2


In [ ]:
def one_hot(Y):
    one_hot_Y=np.zeros((Y.size,Y.max()+1))
    one_hot_Y[np.arange(Y.size),Y]=1
    return one_hot_Y.T
def deriv_ReLU(Z):
    # can have plausible 2 values only 0,1  so....
    return Z>0 
def propogation_context_bckwrd(Z1,A1,A2,W2,Z2,Y,X):
    m = Y.size  # Use actual sample size consistently
    one_hot_Y=one_hot(Y)
    dZ2 = A2-one_hot_Y
    dW2 = (1/m) * dZ2.dot(A1.T)          # Use m consistently
    dB2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2)*deriv_ReLU(Z1)
    dW1 = (1/m) * dZ1.dot(X.T)           # Use m consistently
    dB1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
    #dZ1 , dZ2 being helping functions are not needed anymore
    return dW1,dB2,dB1,dW2
def update_params(W1,B1,W2,B2,dW1,dB1,dW2,dB2,alpha):
    W1= W1-alpha*dW1
    B1= B1-alpha*dB1
    W2= W2-alpha*dW2
    B2= B2-alpha*dB2
    return W1,B1,W2,B2  

In [ ]:
def get_predictions(A2):
    return np.argmax(A2,0)
def get_accuracy(predictions,Y):
    print(predictions,Y)
    return np.sum(predictions==Y)/Y.size


def gradient_descent_new(X,Y,iterations,alpha):
    W1,B1,W2,B2=init_params()
    for i in range(iterations):
        Z1,A1,Z2,A2 = propogation_context_frwd(W1,B1,W2,B2,X)
        dW1,dB2,dB1,dW2= propogation_context_bckwrd(Z1,A1,A2,W2,Z2,Y,X)
        W1,B1,W2,B2=update_params(W1,B1,W2,B2,dW1,dB1,dW2,dB2,alpha)  # Fixed order
        if i%50==0:
            print("Iteration: ", i)
            print("Accuracy:" ,get_accuracy(get_predictions(A2),Y))
    return W1,B1,W2,B2

# **Acheived 97% accuracy with ordered dataset MNIST Digit Recognizer and almost 96% with unordered Digit Recognizer data set.**

In [ ]:
W1,B1,W2,B2=gradient_descent_new(X_train, Y_train, 1000, 0.3)